In [1]:
# === Presentation mode — run this cell once to enlarge fonts, style the notebook & enable auto-scroll for live demo ===
from IPython.display import HTML, display

display(HTML('''
<style>
/* Enlarge markdown text */
.jp-RenderedMarkdown, .rendered_html { font-size: 17px !important; line-height: 1.55 !important; }
.jp-RenderedMarkdown h1, .rendered_html h1 { font-size: 2.4em !important; color: #D42127; border-bottom: 3px solid #D42127; padding-bottom: 8px; }
.jp-RenderedMarkdown h2, .rendered_html h2 { font-size: 1.9em !important; color: #fff; background: linear-gradient(90deg, #D42127 0%, #5a0e10 100%); padding: 10px 18px; border-radius: 6px; }
.jp-RenderedMarkdown h3, .rendered_html h3 { font-size: 1.5em !important; color: #D42127; border-left: 5px solid #D42127; padding-left: 12px; }
.jp-RenderedMarkdown h4, .rendered_html h4 { font-size: 1.25em !important; color: #333; }

/* Enlarge code in markdown and code cells */
.jp-RenderedMarkdown code, .rendered_html code { font-size: 16px !important; background: #fff3f3; padding: 2px 6px; border-radius: 3px; color: #b00020; }
.jp-RenderedMarkdown pre, .rendered_html pre { font-size: 15px !important; }
.CodeMirror, .jp-InputArea-editor { font-size: 16px !important; }

/* Enlarge output text */
.jp-OutputArea-output pre, .output_subarea pre { font-size: 14px !important; line-height: 1.4 !important; }

/* Highlight blockquotes (the > notes) */
.jp-RenderedMarkdown blockquote, .rendered_html blockquote {
    border-left: 5px solid #D42127; background: #fff8f8; padding: 12px 18px; margin: 12px 0;
    font-size: 17px; border-radius: 0 6px 6px 0;
}

/* Bigger tables */
.jp-RenderedMarkdown table, .rendered_html table { font-size: 16px !important; }
.jp-RenderedMarkdown th, .rendered_html th { background: #D42127 !important; color: white !important; padding: 8px 12px !important; }
.jp-RenderedMarkdown td, .rendered_html td { padding: 6px 12px !important; }

/* Make images responsive & larger by default */
.jp-OutputArea-output img, .output_subarea img { max-width: 95% !important; }
</style>
'''))

display(HTML('''
<script>
// Auto-scroll all cell outputs to bottom as new content arrives
(function() {
    const observer = new MutationObserver(mutations => {
        mutations.forEach(m => {
            const output = m.target.closest('.jp-OutputArea-output, .output_subarea');
            if (output) {
                const container = output.closest('.jp-OutputArea, .output');
                if (container) container.scrollTop = container.scrollHeight;
            }
        });
    });
    observer.observe(document.body, {childList: true, subtree: true, characterData: true});
    console.log('Auto-scroll enabled');
})();
</script>
'''))
print('Presentation styling applied — fonts enlarged, headers styled, auto-scroll enabled.')

Presentation styling applied — fonts enlarged, headers styled, auto-scroll enabled.


# AMD Ryzen™ AI Software — NPU Inference on Linux Walkthrough
**2026 DSP & AI Summit · Thomas Zerbs**

---

## End-to-End Flow

| Step | What | Tool |
|:----:|------|------|
| 1 | Load pre-trained PyTorch model | `torchvision` |
| 2 | Export to ONNX | `torch.onnx.export()` |
| 3 | Quantize | AMD Quark (INT8) · VAIML auto-cast (BF16) |
| 4 | Compile, cache & infer | ONNX Runtime + Vitis AI EP |
| 5 | Analyze | AI Analyzer · `xrt-smi` |

> **System**: HP Z2 Mini PC (Strix Halo APU) · Ubuntu 24.04.3 · Kernel 6.17 · Ryzen AI SW 1.7.1

## Walkthrough Agenda

| Tutorial | Model | Focus |
|---|---------|-----------------|
| **A** | ResNet50 INT8 | Quark PTQ → INT8 ONNX → NPU inference |
| **B** | ResNet50 BF16 | VAIML auto-cast FP32→BF16 → NPU inference |
| **C** | YOLOv8m | Detection + COCO evaluation (CPU vs NPU) + AI Analyzer |
| **D** | Benchmarking & `xrt-smi` | Performance matrix across all models |

### Prerequisites
**The RyzenAI-SW repository is already cloned and all dependencies are pre-installed.**  
Run the cell below to verify key packages are available.

In [2]:
%%bash
echo "=== INT8 Tutorial Requirements ==="
cat /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8/requirements.txt
echo ""
echo "=== BF16 Tutorial Requirements ==="
cat /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/bf16/requirements.txt
echo ""
echo "=== Key Installed Packages ==="
pip show onnxruntime onnxruntime-vitisai quark torchvision numpy opencv-python 2>/dev/null | grep -E '^(Name|Version):' 2>&1
echo ""
echo "=== AI Analyzer Version ==="
aianalyzer --version 2>&1

=== INT8 Tutorial Requirements ===
torchvision==0.23.0
opencv-python==4.11.0.86
numpy==1.26.4

=== BF16 Tutorial Requirements ===
timm==1.0.20
torch==2.8.0
torchvision==0.23.0
opencv-python==4.11.0.86
numpy==1.26.4
=== Key Installed Packages ===
Name: onnxruntime-vitisai
Version: 1.24.1
Name: torchvision
Version: 0.20.1+cpu
Name: numpy
Version: 1.26.4
Name: lapack-lite
Name: tempita
Name: dragon4
Name: libdivide
Name: Meson
Name: spin
Name: OpenBLAS
Name: LAPACK
Name: GCC runtime library
Name: libquadmath
Name: opencv-python
Version: 4.11.0.86

=== AI Analyzer Version ===
1.7.0.dev20260130181427+g301504b8


In [3]:
import os, sys
from pathlib import Path

REPO     = Path('/scratch/thozerbs/git/RyzenAI-SW')
INT8_DIR = REPO / 'CNN-examples/getting_started_resnet/int8'
BF16_DIR = REPO / 'CNN-examples/getting_started_resnet/bf16'
YOLO_DIR = REPO / 'CNN-examples/object_detection/yolov8m'
BENCH_DIR = REPO / 'onnx-benchmark'

os.chdir(INT8_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8


In [4]:
# Verify NPU is detected — always run this first
!xrt-smi examine

System Configuration
  OS Name              : Linux
  Release              : 6.17.0-19-generic
  Machine              : x86_64
  CPU Cores            : 32
  Memory               : 96323 MB
  Distribution         : Ubuntu 24.04.3 LTS
  GLIBC                : 2.39
  Model                : HP Z2 Mini G1a Workstation Desktop PC
  BIOS Vendor          : HP
  BIOS Version         : X53 Ver. 01.02.03
  Processor            : AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S

XRT
  Version              : 2.21.75
  Branch               : HEAD
  Hash                 : 4eb1f4392a012b4e6eca759762389c612537f7c7
  Hash Date            : 2026-03-09 20:30:37
  amdxdna Version      : 2.21.260102.53.release_20260309, 6f881ad230142b707ca8ce5b33fca426a926c551
  virtio-pci Version   : 6.17.0-19-generic
  NPU Firmware Version : 1.1.2.65

Device(s) Present
|BDF             |Name            |
|----------------|----------------|
|[0000:c5:00.1]  |NPU Strix Halo  |




---

<div style='background:linear-gradient(90deg,#D42127,#8B0000);color:white;padding:20px;border-radius:8px;margin:20px 0;text-align:center;'><h1 style='color:white;border:none;margin:0;'>Tutorial A · ResNet50 INT8</h1><div style='font-size:18px;opacity:0.9;margin-top:6px;'>Quark PTQ → XINT8 ONNX → VitisAI EP</div></div>

## Tutorial A — ResNet50 INT8
**Prepare Data · Load Model · Quantize · Deploy**  
`PyTorch` · `AMD Quark` · `XINT8` · `Vitis AI EP` · `ONNX Runtime`

### A1 — Load Data, Model & Export to ONNX

`prepare_model_data.py` automates three steps: download CIFAR-10, load pre-trained ResNet50, and export to ONNX.

**Key APIs** (`prepare_model_data.py`):

```python
from torchvision.models import resnet50, ResNet50_Weights

# 1. Load ResNet50 with pre-trained ImageNet weights
weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)

# Replace FC head: ImageNet has 1,000 classes, CIFAR-10 has 10
# Keep the full backbone, swap only the head: 2048 → 64 → 10
resnet.fc = nn.Sequential(nn.Linear(2048, 64), nn.ReLU(inplace=True), nn.Linear(64, 10))
```

```python
# 2. Export to ONNX
dummy_inputs = torch.randn(1, 3, 32, 32)       # CIFAR-10 input size
torch.onnx.export(
    model, dummy_inputs,
    'models/resnet_trained_for_cifar10.onnx',
    opset_version=17,                           # VitisAI EP / VAIML requires opset 17
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
```

> **ONNX opset 17** is required by VitisAI EP / VAIML. If your model uses a different opset version, convert it using the ONNX Version Converter.

**Output**: `models/resnet_trained_for_cifar10.onnx` — FP32 ONNX, ready for Quark quantization (INT8) or VAIML compilation (BF16).

In [5]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8
python -u prepare_model_data.py 2>&1

### A2 — Quantize with AMD Quark (INT8)

`resnet_quantize.py` — post-training quantization (PTQ). Quark converts the FP32 ONNX model to INT8 ONNX.

**Key APIs** (`resnet_quantize.py`):

```python
from quark.onnx.quantization.config import Config, get_default_config
from quark.onnx import ModelQuantizer

# 1. Pick a quantization config (XINT8 | A8W8 | A16W8 | BF16 | BFP16)
quant_config = get_default_config('XINT8')        # Symmetric INT8, power-of-two scales
config = Config(global_quant_config=quant_config)

# 2. Run post-training quantization with calibration data
quantizer = ModelQuantizer(config)
quantizer.quantize_model(
    input_model_path  = 'models/resnet_trained_for_cifar10.onnx',   # FP32 ONNX
    output_model_path = 'models/resnet_quantized.onnx',            # INT8 ONNX
    resnet_calibration_reader(calibration_dataset_path)             # calibration data reader
)
```

The `CalibrationDataReader` feeds representative CIFAR-10 images to calibrate activation ranges.

**Output**: `models/resnet_quantized.onnx` — standard ONNX, ready for VitisAI EP.


#### Improving Post-Quantization Accuracy

Quark's default `XINT8` config is fast but may leave accuracy on the table. To improve:

| Method | Description | When to use |
|--------|-------------|-------------|
| **Calibration** | Feed representative data to set activation ranges (already doing this) | Always |
| **CLE** (Cross-Layer Equalization) | Balances weight ranges across layers | Models with high inter-layer weight variance |
| **Bias correction** | Compensates for quantization-induced bias | Minimal extra cost, usually helpful |
| **AdaRound** | Learns optimal rounding per weight via layer-wise optimization | Best accuracy, takes minutes–hours |
| **AdaQuant** | Jointly optimizes weights + quantization params (mutually exclusive with AdaRound) | Alternative to AdaRound |

L2 loss vs float baseline (ResNet-50, from Quark docs):

| | No calibration | + Calibration (A8W8) | + AdaRound | + AdaQuant |
|---|---|---|---|---|
| **L2 loss** | 30.26 | 9.78 | 1.43 | **1.15** |

> **Reference**: [Advanced Quantization Quick Start for Ryzen AI — AMD Quark 0.11.1 docs](https://quark.docs.amd.com/latest/supported_accelerators/ryzenai/tutorial_quick_start_for_ryzenai.html) — swap `XINT8` for `A8W8_ADAROUND` or `A8W8_ADAQUANT` in `get_default_config()` to enable.


In [6]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8
python -u resnet_quantize.py 2>&1


[QUARK-INFO]: Checking custom ops library ...

[QUARK-INFO]: The CPU version of custom ops library already exists.

[QUARK-INFO]: Checked custom ops library.
The configuration for quantization is Config(global_quant_config=QuantizationConfig(calibrate_method=<PowerOfTwoMethod.MinMSE: 1>, quant_format=<QuantFormat.QDQ: 1>, activation_type=<QuantType.QUInt8: 1>, weight_type=<QuantType.QInt8: 0>, input_nodes=[], output_nodes=[], op_types_to_quantize=[], nodes_to_quantize=[], extra_op_types_to_quantize=[], nodes_to_exclude=[], subgraphs_to_exclude=[], specific_tensor_precision=False, execution_providers=['CPUExecutionProvider'], per_channel=False, reduce_range=False, optimize_model=True, use_dynamic_quant=False, use_external_data_format=False, convert_fp16_to_fp32=False, convert_nchw_to_nhwc=False, include_sq=False, include_rotation=False, include_cle=True, include_auto_mp=False, include_fast_ft=False, enable_npu_cnn=True, enable_npu_transformer=False, debug_mode=False, crypto_mode=False,

In [7]:
# Show Quark quantization summary — timing breakdown + per-node dtype info
summary_path = INT8_DIR / 'quantized_info.csv'
print(summary_path.read_text())

models/resnet_trained_for_cifar10.onnx quantization quantization info,2026-04-14 16:00:55
quantization stage,time consumed(s),sub stage,time consumed(s)
pre process,0.90
,,calibration: collect data (onnx inference + numpy statistics),3.54
,,calibration: compute data,8.43
calibration (collect data + compute data),12.24
static quantization,0.43
post process(including finetuning),0.00
e2e,14.65089714503847

Node Name,Op Type,Activation,Weights,Bias
/conv1/Conv,Conv,UINT8,INT8,INT8
/maxpool/MaxPool,MaxPool,UINT8,,
/layer1/layer1.0/conv1/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.0/downsample/downsample.0/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.0/conv2/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.0/conv3/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.0/Add,Add,UINT8,,
/layer1/layer1.1/conv1/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.1/conv2/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.1/conv3/Conv,Conv,UINT8,INT8,INT8
/layer1/layer1.1/Add,Add,UINT8,,
/layer1/layer1.2/conv1/Conv,Conv,UINT8,INT8,INT8
/layer1/

### A3 — Inference: CPU Baseline → NPU

`predict.py` creates an ONNX Runtime `InferenceSession` and runs predictions on 10 CIFAR-10 test images.  
**Best practice**: run CPU first to verify the model is correct, then switch to NPU.

**Key APIs** (`predict.py`):

```python
import onnxruntime as ort

# --- CPU (default, no flag) ---
providers        = ['CPUExecutionProvider']
provider_options = [{}]

# --- NPU via Vitis AI EP (--ep npu) ---
providers = ['VitisAIExecutionProvider']
provider_options = [{
    'cache_dir': str(cache_dir),       # compiled NPU binary saved here
    'cache_key': 'modelcachekey',      # change to force recompile when model changes
    'enable_cache_file_io_in_mem': '0',
}]

# Create ONNX RT session
session_options = ort.SessionOptions()
session_options.log_severity_level = 1  # 0=Verbose 1=Info 2=Warning 3=Error 4=Fatal

session = ort.InferenceSession(
    model.SerializeToString(),
    sess_options=session_options,
    providers=providers,
    provider_options=provider_options
)

# Run inference — identical API regardless of CPU or NPU
outputs = session.run(None, {'input': input_data})
predicted_class = np.argmax(outputs[0])
```

**Provider options for INT8 on STX/KRK** (focused):
- `cache_dir` & `cache_key` — compiled binary cache location and unique model key
- `opt_level` — compiler optimization level (0, 1, 2, 3, 65536), INT8 only
- `enable_cache_file_io_in_mem` — keep compiled model in memory (1) or save to disk (0)
- If compiling INT8 on **PHX/HPT**: `xclbin` & `target` must be set

#### Full Vitis AI EP Provider Options

| Option | Description |
|--------|-------------|
| `config_file` | Path to compilation config JSON (required for BF16 compilation) |
| `target` | Compiler backend — `X2` (STX/KRK, default) or `X1` (PHX/HPT legacy integer) |
| `xclbin` | xclbin path — required for INT8 on PHX/HPT |
| `cache_dir` | Directory where compiled NPU artifacts are stored |
| `cache_key` | Subfolder name within `cache_dir` for this model's compiled artifacts |
| `enable_cache_file_io_in_mem` | `1` = keep compiled model in memory, `0` = save to disk |
| `opt_level` | Compiler optimization level (0, 1, 2, 3, 65536) — INT8 only |
| `encryption_key` | 256-bit key for encrypting the EP context model |
| `ai_analyzer_visualization` | Enable compilation-time analysis data collection |
| `ai_analyzer_profiling` | Enable inference-time analysis data collection |

Source: [Model Compilation and Deployment — Ryzen AI Software 1.7.1](https://ryzenai.docs.amd.com/en/latest/modelrun.html)

> **First NPU run** compiles the INT8 model (~60 s) and caches to `./modelcachekey/`. Subsequent runs load from cache in under 1 second. Predictions should match CPU output.


In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8
# Step 1: Run on CPU — verify model correctness (no NPU driver needed)
python -u predict.py 2>&1

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8
# Step 2: Run on NPU with Vitis AI EP
# First run: compiles INT8 model (~60 s), cached to ./modelcachekey/
# Subsequent runs: loads from cache (<1 s)
python -u predict.py --ep npu 2>&1

---

<div style='background:linear-gradient(90deg,#D42127,#8B0000);color:white;padding:20px;border-radius:8px;margin:20px 0;text-align:center;'><h1 style='color:white;border:none;margin:0;'>Tutorial B · ResNet50 BF16</h1><div style='font-size:18px;opacity:0.9;margin-top:6px;'>VAIML auto-cast FP32 → BF16 → VitisAI EP</div></div>

## Tutorial B — ResNet50 BF16
**Vitis AI Compiler Auto-Casting**  
`VAIML` · `BF16`

| Path | Tool | Quantization | Calibration data? |
|------|------|-------------|-------------------|
| **INT8** | AMD Quark | Full INT8 quantization **before** compilation | Yes — required |
| **BF16** | Vitis AI Compiler (VAIML) | Auto-casts FP32 → BF16 **during** compilation | No — not needed |

> *Quark does support BF16, however the VAIML automatic conversion is the integrated Ryzen AI path: BF16 lowering happens together with NPU partitioning in ONNX Runtime + Vitis AI EP, and without calibration data.*

Uses the same FP32 ONNX model from Tutorial A.

In [ ]:
os.chdir(BF16_DIR)
print(f'Working directory: {os.getcwd()}')

### B1 — Compile FP32 Model to BF16

`compile.py` — VAIML auto-casts FP32 weights and activations to BF16, saves NPU binary artifacts to cache.  
Compilation is triggered by creating an `InferenceSession` — no separate compile step needed.

**`vitisai_config.json`** — the key field is `enable_f32_to_bf16_conversion: true`:
```json
{
    "passes": [
        { "name": "init", "plugin": "vaip-pass_init" },
        { "name": "vaiml_partition", "plugin": "vaip-pass_vaiml_partition",
          "vaiml_config": { "enable_f32_to_bf16_conversion": true } }
    ],
    "target": "VAIML"
}
```

**Key APIs** (`compile.py`):

```python
provider_options_dict = {
    'config_file':                config_file,   # points to vitisai_config.json
    'cache_dir':                  cache_dir,     # NPU binary artifacts saved here
    'cache_key':                  cache_key,     # change when model changes to recompile
    'enable_cache_file_io_in_mem': 0,
    'ai_analyzer_visualization':  True,          # graph partition + op fusion (JSON)
    'ai_analyzer_profiling':      True,          # per-operator timing (JSON)
}

# Compilation triggered by creating the InferenceSession
session = onnxruntime.InferenceSession(
    onnx_model,
    providers=['VitisAIExecutionProvider'],
    provider_options=[provider_options_dict]
)
```

#### Full Vitis AI EP Provider Options

| Option | Description |
|--------|-------------|
| `config_file` | Path to compilation config JSON (required for BF16 compilation) |
| `target` | Compiler backend — `X2` (STX/KRK, default) or `X1` (PHX/HPT legacy integer) |
| `xclbin` | xclbin path — required for INT8 on PHX/HPT |
| `cache_dir` | Directory where compiled NPU artifacts are stored |
| `cache_key` | Subfolder name within `cache_dir` for this model's compiled artifacts |
| `enable_cache_file_io_in_mem` | `1` = keep compiled model in memory, `0` = save to disk |
| `opt_level` | Compiler optimization level (0, 1, 2, 3, 65536) — INT8 only |
| `encryption_key` | 256-bit key for encrypting the EP context model |
| `ai_analyzer_visualization` | Enable compilation-time analysis data collection |
| `ai_analyzer_profiling` | Enable inference-time analysis data collection |

Source: [Model Compilation and Deployment — Ryzen AI Software 1.7.1](https://ryzenai.docs.amd.com/en/latest/modelrun.html)

**Output**: NPU-optimized binary artifacts in `my_cache_dir/`


In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/bf16
# Compile FP32 ONNX → BF16 NPU binary (~60 seconds)
# Uses the same FP32 ONNX model exported in Tutorial A
python -u compile.py --model ../int8/models/resnet_trained_for_cifar10.onnx 2>&1

In [ ]:
# Show VAIML compilation summary — what was offloaded to the NPU
summary_path = BF16_DIR / 'my_cache_dir' / 'resnet_trained_for_cifar10' / 'final-vaiml-pass-summary.txt'
print(summary_path.read_text())

### B2a — BF16 Inference on CPU

Run the FP32 model on CPU as a baseline (standard `CPUExecutionProvider`, no config file needed).

```python
# CPU path — no config_file, no VitisAI EP
providers = ['CPUExecutionProvider']
provider_options_dict = {}
```

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/bf16
# Run BF16 predict on CPU — baseline comparison
python -u predict.py --ep cpu 2>&1

### B2b — BF16 Inference on NPU

`predict.py` (BF16) loads compiled artifacts from cache and runs inference on 1000 CIFAR-10 images.

**Key APIs** (`predict.py` BF16 path):

```python
# BF16 provider options — config_file is REQUIRED (INT8 does not need it)
provider_options_dict = {
    'config_file':                config_file,  # required for BF16 / VAIML path
    'cache_dir':                  cache_dir,    # loads compiled NPU binary from here
    'cache_key':                  cache_key,    # must match key used during compile
    'enable_cache_file_io_in_mem': 0,
    'ai_analyzer_visualization':  True,
    'ai_analyzer_profiling':      True,
}

session_options.enable_profiling = True          # ORT profiler for CPU layers (AI Analyzer)

session = onnxruntime.InferenceSession(
    onnx_model_path,
    sess_options=session_options,
    providers=['VitisAIExecutionProvider'],
    provider_options=[provider_options_dict]
)

outputs = session.run(None, {input_name: input_data})
session.end_profiling()   # flush ORT profiler JSON for AI Analyzer
```

> `session.end_profiling()` flushes the ORT native profiler data to a JSON file — this captures CPU-side layer timings for the AI Analyzer Performance panel.

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/bf16
# Run BF16 inference on NPU — loads from cache (<1 s)
python -u predict.py --ep npu 2>&1

---

<div style='background:linear-gradient(90deg,#D42127,#8B0000);color:white;padding:20px;border-radius:8px;margin:20px 0;text-align:center;'><h1 style='color:white;border:none;margin:0;'>Tutorial C · YOLOv8m</h1><div style='font-size:18px;opacity:0.9;margin-top:6px;'>Object detection · COCO evaluation · AI Analyzer</div></div>

## Tutorial C — YOLOv8m
**Inference · Evaluation · Analysis**  
`ONNX Runtime` · `COCO` · `AI Analyzer`

Uses YOLOv8m BF16 (pre-compiled) to demonstrate:
1. **Inference** — single-image object detection on NPU
2. **COCO Evaluation** — accuracy comparison (CPU vs NPU)
3. **AI Analyzer** — visualize NPU/CPU partitioning and per-operator profiling

In [ ]:
os.chdir(YOLO_DIR)
print(f'Working directory: {os.getcwd()}')

### C1 — Single-Image Inference (NPU BF16)

Run YOLOv8m BF16 on a test image using the NPU. The output image shows bounding boxes with class labels and confidence scores.

```python
provider_options = [{
    'config_file':               'vaiml_config.json',
    'cache_dir':                 str(Path('.').resolve()),
    'cache_key':                 'modelcachekey',
    'ai_analyzer_visualization': True,
    'ai_analyzer_profiling':     True,
}]

session = ort.InferenceSession(onnx_path,
    sess_options=session_options,
    providers=['VitisAIExecutionProvider'],
    provider_options=provider_options)
```

In [ ]:
import subprocess
from IPython.display import Image, display

YOLO_DIR = '/scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m'

# Run YOLOv8m BF16 on NPU — generates AI Analyzer JSON artifacts
subprocess.run(
    ['python', '-u', 'run_inference.py',
     '--model_input', 'models/yolov8m_BF16.onnx',
     '--input_image', 'test_image.jpg',
     '--output_image', 'test_output.jpg',
     '--device', 'npu-bf16'],
    cwd=YOLO_DIR,
    stderr=subprocess.STDOUT
)

# Display the detection output
display(Image(filename=f'{YOLO_DIR}/test_output.jpg', width=800))

### C2 — COCO Evaluation: CPU vs NPU Accuracy

Evaluate the BF16 model on 500 COCO validation images to compare accuracy between CPU and NPU.
The script reports **mAP**, **mAP50**, and **mAP75** and saves:
- `coco-metrics.json` — accuracy metrics
- Predicted images with bounding boxes to `runs/onnx-predict/`

#### CPU Evaluation

In [ ]:
import subprocess, json, os, glob
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

YOLO_DIR = '/scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m'
EVAL_DIR = '/scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m/runs/onnx-predict/yolov8m_BF16-instances_val2017-iou=0.50'

# Run COCO evaluation on CPU (500 images)
subprocess.run(
    ['python', '-u', 'run_inference.py',
     '--model_input', 'models/yolov8m_BF16.onnx',
     '--evaluate',
     '--coco_dataset', 'datasets/coco',
     '--device', 'cpu',
     '--eval_max_images', '500'],
    cwd=YOLO_DIR,
    stderr=subprocess.STDOUT
)

# Load metrics
metrics_file = os.path.join(EVAL_DIR, 'coco-metrics.json')
with open(metrics_file) as f:
    cpu_metrics = json.load(f)

# --- Radar Chart: mAP by object size + IoU thresholds ---
radar_keys = ['mAP', 'mAP50', 'mAP75', 'mAP_small', 'mAP_medium', 'mAP_large']
radar_labels = ['mAP\n(0.50:0.95)', 'mAP50', 'mAP75', 'mAP\nSmall', 'mAP\nMedium', 'mAP\nLarge']
values = [cpu_metrics[k] * 100 for k in radar_keys]
angles = np.linspace(0, 2 * np.pi, len(radar_keys), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw=dict(polar=True))
fig.suptitle('CPU Evaluation — COCO Metrics', fontsize=14, fontweight='bold', y=1.02)

# Radar
ax = axes[0]
ax.fill(angles, values, alpha=0.25, color='#1f77b4')
ax.plot(angles, values, 'o-', color='#1f77b4', linewidth=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=9)
ax.set_ylim(0, 100)
ax.set_title('Detection Accuracy Profile', pad=20, fontsize=11)
for a, v in zip(angles[:-1], values[:-1]):
    ax.annotate(f'{v:.1f}%', xy=(a, v), fontsize=8, ha='center', va='bottom')

# Bar chart of all metrics
ax2 = axes[1]
ax2.set_axis_off()  # remove polar
ax2 = fig.add_subplot(1, 2, 2)  # replace with regular axes
bar_keys = ['mAP', 'mAP50', 'mAP75', 'mAP_small', 'mAP_medium', 'mAP_large', 'AR@1', 'AR@10', 'AR@100', 'AR_small', 'AR_medium', 'AR_large']
bar_labels = ['mAP', 'mAP50', 'mAP75', 'mAP Small', 'mAP Med', 'mAP Large', 'AR@1', 'AR@10', 'AR@100', 'AR Small', 'AR Med', 'AR Large']
bar_vals = [cpu_metrics[k] * 100 for k in bar_keys]
colors = ['#1f77b4']*6 + ['#2ca02c']*6
y_pos = range(len(bar_keys))
ax2.barh(y_pos, bar_vals, color=colors, height=0.7)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(bar_labels, fontsize=9)
ax2.set_xlim(0, 100)
ax2.set_xlabel('Score (%)')
ax2.set_title('All COCO Metrics', fontsize=11)
ax2.invert_yaxis()
for i, v in enumerate(bar_vals):
    ax2.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# Show sample predicted images
pred_images = sorted(glob.glob(os.path.join(EVAL_DIR, 'predict_of_*.png')))[:3]
for img_path in pred_images:
    print(f'\n{os.path.basename(img_path)}:')
    display(Image(filename=img_path, width=600))

#### NPU Evaluation

In [ ]:
import subprocess, json, os, glob
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

YOLO_DIR = '/scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m'
EVAL_DIR = '/scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m/runs/onnx-predict/yolov8m_BF16-instances_val2017-iou=0.50'

# Run COCO evaluation on NPU (500 images)
subprocess.run(
    ['python', '-u', 'run_inference.py',
     '--model_input', 'models/yolov8m_BF16.onnx',
     '--evaluate',
     '--coco_dataset', 'datasets/coco',
     '--device', 'npu-bf16',
     '--eval_max_images', '500'],
    cwd=YOLO_DIR,
    stderr=subprocess.STDOUT
)

# Load NPU metrics
metrics_file = os.path.join(EVAL_DIR, 'coco-metrics.json')
with open(metrics_file) as f:
    npu_metrics = json.load(f)

# --- Radar Chart: NPU accuracy profile ---
radar_keys = ['mAP', 'mAP50', 'mAP75', 'mAP_small', 'mAP_medium', 'mAP_large']
radar_labels = ['mAP\n(0.50:0.95)', 'mAP50', 'mAP75', 'mAP\nSmall', 'mAP\nMedium', 'mAP\nLarge']
values = [npu_metrics[k] * 100 for k in radar_keys]
angles = np.linspace(0, 2 * np.pi, len(radar_keys), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

fig, ax_radar = plt.subplots(1, 1, figsize=(6, 5), subplot_kw=dict(polar=True))
fig.suptitle('NPU Evaluation — Detection Accuracy Profile', fontsize=13, fontweight='bold', y=1.02)
ax_radar.fill(angles, values, alpha=0.25, color='#ff7f0e')
ax_radar.plot(angles, values, 'o-', color='#ff7f0e', linewidth=2)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, fontsize=9)
ax_radar.set_ylim(0, 100)
for a, v in zip(angles[:-1], values[:-1]):
    ax_radar.annotate(f'{v:.1f}%', xy=(a, v), fontsize=8, ha='center', va='bottom')
plt.tight_layout()
plt.show()

# --- Side-by-side Grouped Bar Chart: CPU vs NPU ---
compare_keys = ['mAP', 'mAP50', 'mAP75', 'AR@100']
compare_labels = ['mAP\n(0.50:0.95)', 'mAP50\n(IoU=0.50)', 'mAP75\n(IoU=0.75)', 'AR@100']
cpu_vals = [cpu_metrics[k] * 100 for k in compare_keys]
npu_vals = [npu_metrics[k] * 100 for k in compare_keys]

x = np.arange(len(compare_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_cpu = ax.bar(x - width/2, cpu_vals, width, label='CPU (FP32)', color='#1f77b4', edgecolor='white')
bars_npu = ax.bar(x + width/2, npu_vals, width, label='NPU (BF16)', color='#ff7f0e', edgecolor='white')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('CPU vs NPU Accuracy Comparison — YOLOv8m BF16', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(compare_labels, fontsize=10)
ax.set_ylim(0, max(max(cpu_vals), max(npu_vals)) * 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Value labels on bars
for bar in bars_cpu:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars_npu:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Show sample predicted images
pred_images = sorted(glob.glob(os.path.join(EVAL_DIR, 'predict_of_*.png')))[:3]
for img_path in pred_images:
    print(f'\n{os.path.basename(img_path)}:')
    display(Image(filename=img_path, width=600))

### C3 — AI Analyzer

**Purpose**: Visualize and analyze model compilation and inference on Vitis AI — understand NPU/CPU partitioning and identify performance bottlenecks.

Two provider option flags enable AI Analyzer data collection:

| Flag | Generates | Powers |
|------|-----------|--------|
| `ai_analyzer_visualization: True` | JSON: graph partition, op fusion | **Partitioning** + **NPU Insights** panels |
| `ai_analyzer_profiling: True` | JSON: per-operator timing (NPU) | **Performance** panel |
| `session_options.enable_profiling = True` (ORT) | JSON: CPU-side layer timings | Full CPU+NPU pipeline in Performance panel |

**Key APIs** (`run_inference.py`):

```python
provider_options = [{
    'config_file':               'vaiml_config.json',
    'cache_dir':                 str(Path('.').resolve()),
    'cache_key':                 'modelcachekey',
    'ai_analyzer_visualization': True,    # graph partition + op fusion artifacts
    'ai_analyzer_profiling':     True,    # per-operator timing artifacts
}]

session_options.enable_profiling = True    # ORT profiler for CPU layers

session = ort.InferenceSession(onnx_path,
    sess_options=session_options,
    providers=['VitisAIExecutionProvider'],
    provider_options=provider_options)

outputs = session.run(None, {input_name: input_img})
session.end_profiling()   # flush ORT profiler JSON
```

> **Note**: AI Analyzer flags must be set when **compiling** (first run) *and* when running **inference**.

In [ ]:
# Launch AI Analyzer — opens in browser at http://localhost:8000
# Interrupt kernel (Ctrl+C) to stop the server when done
!aianalyzer .

---

<div style='background:linear-gradient(90deg,#D42127,#8B0000);color:white;padding:20px;border-radius:8px;margin:20px 0;text-align:center;'><h1 style='color:white;border:none;margin:0;'>Tutorial D · Benchmarking & NPU Management</h1><div style='font-size:18px;opacity:0.9;margin-top:6px;'>Performance · xrt-smi · onnx-benchmark</div></div>

## Tutorial D — Benchmarking & NPU Management
**Performance · `xrt-smi` · `onnx-benchmark`**

### D1 — NPU Management with `xrt-smi`

`xrt-smi` is the NPU management utility — analogous to `nvidia-smi` for NVIDIA GPUs.

| Command | Purpose |
|---------|----------|
| `xrt-smi examine` | System info: OS, XRT version, NPU driver/firmware, device BDF and name |
| `xrt-smi examine --report platform` | Performance mode and estimated power draw (Watts) |
| `xrt-smi examine --report aie-partitions` | Run **while a model is active** — NPU partition and column occupancy |
| `xrt-smi validate --run all` | Standalone sanity tests: no-op latency, DMA throughput, gemm INT8 TOPS |
| `xrt-smi configure --pmode performance` | Set performance mode (powersaver \| balanced \| performance \| turbo) |

In [ ]:
# System info: OS, XRT version, NPU driver, device BDF and name
!xrt-smi examine

In [ ]:
# Performance mode and estimated power draw (Watts)
!xrt-smi examine --report platform

In [ ]:
# NPU partition details and column occupancy
# Best run in a second terminal WHILE a model is actively running on the NPU
!xrt-smi examine --report aie-partitions

In [ ]:
# Standalone NPU sanity tests: no-op latency, DMA throughput, gemm INT8 TOPS
!xrt-smi validate --run all --verbose

In [ ]:
# Set NPU to performance mode before benchmarking
!xrt-smi configure --pmode performance

### D2 — Benchmarking

Two benchmarking approaches:

#### Built-in: `--benchmark` flag in `run_inference.py`
Measures end-to-end app performance (pre/post-processing included). Model-agnostic — works for any model by swapping session + model path.

```python
def benchmark(session, input_name, input_img, num_inference=100):
    # Warmup — 10 runs (driver/JIT settling)
    for _ in range(10):
        session.run(None, {input_name: input_img})

    # Timed inference loop
    time_list = []
    for _ in range(num_inference):
        start = time.time()
        output = session.run(None, {input_name: input_img})
        time_list.append(time.time() - start)

    avg = sum(time_list) / num_inference
    print(f'Avg time: {avg:.3f} s  |  {1/avg:.1f} FPS')
```

#### Standalone: `performance_benchmark.py` (onnx-benchmark tool)
Isolates pure inference with standardized reporting, power analysis, and GUI support.

```bash
python performance_benchmark.py \
    --model_path models/resnet_quantized.onnx \
    --config $RYZEN_AI_.../vaip_config.json \
    --execution_provider VitisAIEP \
    --config $VAIP_CONFIG \
    --autoquant 0 --renew 0 --num 100 --timelimit 10
```

| YOLOv8m | Avg latency | FPS |
|---------|------------|-----|
| Float32 (CPU) | 0.486 s | 2.1 |
| BF16 (NPU) | 0.075 s | 13.4 |
| XINT8 (NPU) | 0.021 s | 48.2 |

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m
# Built-in benchmark: YOLOv8m BF16 on NPU (10 warmup + 100 timed runs)
xrt-smi configure --pmode performance
python -u run_inference.py \
    --model_input models/yolov8m_BF16.onnx \
    --input_image test_image.jpg \
    --output_image test_output.jpg \
    --device npu-bf16 \
    --benchmark 2>&1

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
# Standalone onnx-benchmark tool: ResNet50 INT8 on NPU
# --autoquant 0 = model already quantized, skip Quark
# --renew 0     = reuse compile cache from Tutorial A
python -u performance_benchmark.py \
    --model_path ../CNN-examples/getting_started_resnet/int8/models/resnet_quantized.onnx \
    --execution_provider VitisAIEP \
    --autoquant 0 --renew 0 --num 100 --timelimit 10 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
# CPU baseline for comparison
python -u performance_benchmark.py \
    --model_path ../CNN-examples/getting_started_resnet/int8/models/resnet_quantized.onnx \
    --execution_provider CPU --num 100 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

#### ResNet50 BF16 — NPU

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
python -u performance_benchmark.py \
    --model_path ../CNN-examples/getting_started_resnet/int8/models/resnet_trained_for_cifar10.onnx \
    --execution_provider VitisAIEP \
    --autoquant 0 --renew 0 --num 100 --timelimit 30 --infinite 0 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

#### ResNet50 BF16 — CPU

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
python -u performance_benchmark.py \
    --model_path ../CNN-examples/getting_started_resnet/int8/models/resnet_trained_for_cifar10.onnx \
    --execution_provider CPU --num 100 --timelimit 30 --infinite 0 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

#### YOLOv8m BF16 — NPU

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
python -u performance_benchmark.py \
    --model_path ../CNN-examples/object_detection/yolov8m/models/yolov8m_BF16.onnx \
    --execution_provider VitisAIEP \
    --autoquant 0 --renew 0 --num 100 --timelimit 30 --infinite 0 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

#### YOLOv8m BF16 — CPU

In [ ]:
%%bash
cd /scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark
python -u performance_benchmark.py \
    --model_path ../CNN-examples/object_detection/yolov8m/models/yolov8m_BF16.onnx \
    --execution_provider CPU --num 100 --timelimit 30 --infinite 0 \
    --config /scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/voe-4.0-linux_x86_64/vaip_config.json 2>&1

### Performance Summary

Fill in the results from the benchmark runs above:

| Model | CPU (FPS / Latency) | NPU (FPS / Latency) | Speedup |
|-------|---------------------|----------------------|---------|
| ResNet50 INT8 | ___ FPS / ___ ms | ___ FPS / ___ ms | ___x |
| ResNet50 BF16 | ___ FPS / ___ ms | ___ FPS / ___ ms | ___x |
| YOLOv8m BF16  | ___ FPS / ___ ms | ___ FPS / ___ ms | ___x |

> **Note:** First NPU run includes compilation time. Re-run with `--renew 0` to see cached performance.